### Churn Analysis

In [1]:
# import libraries
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
db_url = os.getenv("DATABASE_URL")

if db_url is None:
    print("Cant find the database URL.")
else:
    engine = create_engine(db_url)
    print("Successfully connected to the database!")



df = pd.read_sql(
    "SELECT * FROM churn_cleaned",
    engine
)

Successfully connected to the database!


In [2]:
# see top 5 rows of the dataframe
df.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn,churn_flag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [3]:
# Create churn flag
df["churn_flag"] = df["churn"].map({
    "Yes": 1,
    "No": 0
})

#### Function to calculate segment impact

In [5]:
def segment_impact(data, segment_column):
    
    result = (
        data.groupby(segment_column)
        .agg(
            total_customers=("customer_id", "count"),
            churned_customers=("churn_flag", "sum"),
            churn_rate=("churn_flag", "mean"),
            revenue_lost=("monthly_charges", 
                          lambda x: x[data.loc[x.index, "churn"]=="Yes"].sum())
        )
        .reset_index()
    )
    
    result["churn_rate"] = (result["churn_rate"] * 100).round(2)
    result["revenue_lost"] = result["revenue_lost"].round(2)
    
    return result

#### Contract wise segment analysis

In [6]:
contract_analysis = segment_impact(
    df,
    "contract"
)

contract_analysis

,contract,total_customers,churned_customers,churn_rate,revenue_lost
0,Month-to-month,3875,1655,42.71,120847.10
1,One year,1473,166,11.27,14118.45
2,Two year,1695,48,2.83,4165.30


#### Tenure wise segment analysis

In [12]:
df["tenure_segment"] = pd.cut(
    df["tenure"],
    bins=[-1,12,36,100],
    labels=[
        "New Customer",
        "Existing Customer",
        "Loyal Customer"
    ]
)

In [14]:
tenure_analysis = segment_impact(
    df,
      "tenure_segment"
)

tenure_analysis

C:\Users\ASUS\AppData\Local\Temp\ipykernel_29352\3377942625.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data.groupby(segment_column)


,tenure_segment,total_customers,churned_customers,churn_rate,revenue_lost
0,New Customer,2186,1037,47.44,68954.25
1,Existing Customer,1856,474,25.54,38249.60
2,Loyal Customer,3001,358,11.93,31927.00


#### Payment method wise segment analysis

In [11]:
payment_analysis = segment_impact(
    df,
    "payment_method"
)

payment_analysis

,payment_method,total_customers,churned_customers,churn_rate,revenue_lost
0,Bank transfer (automatic),1544,258,16.71,20091.90
1,Credit card (automatic),1522,232,15.24,17946.60
2,Electronic check,2365,1071,45.29,84288.75
3,Mailed check,1612,308,19.11,16803.60


Among the analyzed customer segmentation dimensions, contract type represented the largest churn-related revenue impact. Month-to-month customers accounted for the majority of revenue loss ($120.8K), making them the primary target segment for further churn analysis.

#### Month-to-Month Segment Analysis

In [15]:
month_to_month = df[
    df["contract"] == "Month-to-month"
]

#### Drivers Analysis

#### Historical Churn Revenue Loss Analysis

In [18]:
month_churned = month_to_month[
    month_to_month["churn"] == "Yes"
]

drivers = [
    "tech_support",
    "online_security",
    "online_backup",
    "device_protection",
    "payment_method",
    "internet_service"
]

driver_analysis = []

for d in drivers:
    
    # Revenue lost from churned customers
    revenue = (
        month_churned
        .groupby(d)["monthly_charges"]
        .sum()
        .reset_index()
    )
    
    revenue = revenue.rename(
        columns={
            d: "category",
            "monthly_charges": "revenue_lost"
        }
    )
    
    
    # Historical churn rate among all month-to-month customers
    churn_rate = (
        month_to_month
        .groupby(d)["churn_flag"]
        .mean()
        .mul(100)
        .round(2)
        .reset_index()
    )
    
    churn_rate = churn_rate.rename(
        columns={
            d: "category",
            "churn_flag": "historical_churn_rate"
        }
    )
    
    
    # Merge revenue loss and churn rate
    result = revenue.merge(
        churn_rate,
        on="category",
        how="left"
    )
    
    result["driver"] = d
    
    driver_analysis.append(result)


# Combine all drivers
driver_revenue_table = pd.concat(
    driver_analysis,
    ignore_index=True
)


# Final table
driver_revenue_table = driver_revenue_table[
    [
        "driver",
        "category",
        "historical_churn_rate",
        "revenue_lost"
    ]
]


driver_revenue_table.sort_values(
    "revenue_lost",
    ascending=False
)

,driver,category,historical_churn_rate,revenue_lost
3,online_security,No,51.05,102397.55
0,tech_support,No,50.37,102382.25
17,internet_service,Fiber optic,54.61,100482.00
6,online_backup,No,50.31,84890.80
9,device_protection,No,47.58,82715.80
14,payment_method,Electronic check,53.73,77315.60
11,device_protection,Yes,43.57,36133.45
8,online_backup,Yes,38.10,33958.45
16,internet_service,DSL,32.22,18367.25
2,tech_support,Yes,30.70,16467.00


#### Future Revenue Exposure Analysis

In [19]:
# 2. Select priority segment
# Month-to-month customers
month_to_month = df[
    df["contract"] == "Month-to-month"
]


# 3. Historical churn rates (from past behaviour)
# This is used as the risk probability
historical_rates = {}

drivers = [
    "online_security",
    "tech_support",
    "internet_service",
    "online_backup",
    "device_protection",
    "payment_method"
]


for driver in drivers:
    
    rate = (
        month_to_month
        .groupby(driver)["churn_flag"]
        .mean()
        .mul(100)
        .round(2)
    )
    
    for category, value in rate.items():
        historical_rates[(driver, category)] = value



# 4. Current active Month-to-month customers
# These are future risk candidates
active_month = month_to_month[
    month_to_month["churn"] == "No"
]


# 5. Calculate future revenue exposure
future_risk = []


for driver in drivers:
    
    categories = active_month[driver].unique()
    
    for category in categories:
        
        group = active_month[
            active_month[driver] == category
        ]
        
        revenue = group["monthly_charges"].sum()
        customers = len(group)
        
        churn_rate = historical_rates.get(
            (driver, category),
            0
        )
        
        future_risk.append({
            "driver": driver,
            "category": category,
            "historical_churn_rate": churn_rate,
            "customers_at_risk": customers,
            "revenue_exposed": round(revenue,2)
        })


# 6. Create final table
future_risk_table = pd.DataFrame(future_risk)


# 7. Sort by revenue exposure
future_risk_table = future_risk_table.sort_values(
    "revenue_exposed",
    ascending=False
)


future_risk_table

,driver,category,historical_churn_rate,customers_at_risk,revenue_exposed
3,tech_support,No,50.37,1330,93152.00
0,online_security,No,51.05,1288,91001.90
7,internet_service,Fiber optic,54.61,966,84699.10
12,device_protection,No,47.58,1255,84079.45
10,online_backup,No,50.31,1137,77053.75
15,payment_method,Electronic check,53.73,856,61414.95
9,online_backup,Yes,38.10,658,50696.55
13,device_protection,Yes,43.57,540,43670.85
6,internet_service,DSL,32.22,829,43051.20
1,online_security,Yes,29.58,507,36748.40


Customer Retention & Churn Risk Analytics | SQL, Python

Analyzed 7,043 customer records using SQL and Python, identifying a 26.5% overall churn rate and quantifying $139K in monthly recurring revenue at risk from customer attrition.

Identified Month-to-month contracts as the highest-impact segment, accounting for $120.8K in historical monthly revenue loss and a 42.71% churn rate through customer segmentation and financial impact analysis.

Analyzed churn drivers within the priority segment, finding customers without Online Security and Tech Support were associated with over $102K each in historical monthly revenue loss, highlighting key retention opportunities.

Assessed future revenue exposure among active Month-to-month customers, identifying $93K+ exposed revenue from customers lacking Tech Support and $91K+ from customers lacking Online Security based on historical churn patterns.